# Rdakt AI + LangChain Integration

Three ways to add PII anonymization to your LangChain models and agents — pick the tier that fits your use case.

| Tier | API | Interception layer | Best for |
|------|-----|--------------------|----------|
| 1 | `create_http_clients()` | HTTP transport | Full control over model setup |
| 2 | `protect_chat_model()` | HTTP transport | Quick wrap of an existing ChatOpenAI |
| 3 | `RdaktLangChainMiddleware` | Message abstraction | Native LangChain agent middleware |

## Setup

Load a config from `examples/configs/`. Change the path to try different behaviors:

| Config | What it does |
|--------|-------------|
| `minimal.yaml` | Regex detection, in-memory store (default) |
| `audit.yaml` | Detects PII and logs it but forwards unanonymized text |
| `production.yaml` | Regex + NER, SQLite persistence, fail-closed |
| `ner_only.yaml` | spaCy NER without regex (requires `rdakt-ai[ner]`) |
| `redis.yaml` | Redis-backed session store for distributed deployments |

In [ ]:
import os
from pathlib import Path

# Set your API key (or export OPENAI_API_KEY in your shell)
os.environ.setdefault("OPENAI_API_KEY", "sk-your-key-here")

from rdakt_ai.config import load_config

# ── Change this path to switch configs ──
CONFIG_FILE = Path("configs/minimal.yaml")

config = load_config(CONFIG_FILE)
print(f"Loaded config: {CONFIG_FILE}")
print(f"  mode: {config.mode}")
print(f"  pipeline: {config.pipeline}")
print(f"  session store: {config.session_store}")

---
## Tier 1: Low-level — `create_http_clients()`

Returns a `(sync_client, async_client)` tuple of httpx clients wired with Rdakt middleware.
You pass them to the `ChatOpenAI` constructor yourself — maximum control.

In [ ]:
from langchain_openai import ChatOpenAI

from rdakt_ai.integrations.langchain import create_http_clients

sync_client, async_client = create_http_clients(config=config, session_key="tier1-conv")

model_tier1 = ChatOpenAI(
    model="gpt-4o-mini",
    http_client=sync_client,
    http_async_client=async_client,
)

print("Model created with Rdakt transport")
print(f"Sync transport: {type(sync_client._transport).__name__}")
print(f"Async transport: {type(async_client._transport).__name__}")

In [ ]:
from langchain_core.messages import HumanMessage

response = await model_tier1.ainvoke([HumanMessage(content="My email is john.doe@acme.com and my SSN is 123-45-6789")])
print(response.content)

---
## Tier 2: High-level — `protect_chat_model()`

Wraps an existing `ChatOpenAI` model in one call. Uses Pydantic v2 `model_copy`
to inject Rdakt-wired httpx clients — no manual client setup needed.

In [ ]:
from langchain_openai import ChatOpenAI

from rdakt_ai.integrations.langchain import protect_chat_model

# Create a normal model
model = ChatOpenAI(model="gpt-4o-mini")

# Wrap it — returns a new model, original is untouched
model_tier2 = protect_chat_model(model, config=config, session_key="tier2-conv")

print(f"Original model has Rdakt transport: {hasattr(model.http_client, '_transport') and 'Rdakt' in type(model.http_client._transport).__name__ if model.http_client else False}")
print(f"Protected model has Rdakt transport: {'Rdakt' in type(model_tier2.http_client._transport).__name__}")

In [ ]:
from langchain_core.messages import HumanMessage

response = await model_tier2.ainvoke([HumanMessage(content="My phone number is (555) 123-4567 and I live at 742 Evergreen Terrace")])
print(response.content)

---
## Tier 3: Message-level — `RdaktLangChainMiddleware`

Native LangChain agent middleware. Anonymizes message text *before* it reaches the model
and deanonymizes the response *after* — no httpx transport involved.

**Note:** This tier only works with `create_agent()`. For direct `ChatModel.invoke()` or
LCEL chains, use Tier 1 or Tier 2 instead.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from rdakt_ai.integrations.langchain import RdaktLangChainMiddleware

agent_tier3 = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini"),
    tools=[],
    middleware=[RdaktLangChainMiddleware(config=config, session_key="tier3-conv")],
)

print("Agent created with RdaktLangChainMiddleware")

In [ ]:
response = await agent_tier3.ainvoke(
    {"messages": [{"role": "user", "content": "My credit card is 4111-1111-1111-1111 and my name is Jane Smith"}]}
)
print(response["messages"][-1].content)

---
## Multi-turn with session persistence

Pass a `session_key` to reuse the same entity map across turns.
The same PII gets the same token every time — works with any tier.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

from rdakt_ai.integrations.langchain import protect_chat_model
from rdakt_ai.stores import MemoryStore

store = MemoryStore()

model_multi = protect_chat_model(
    ChatOpenAI(model="gpt-4o-mini"),
    config=config,
    session_key="multi-turn",
    store=store,
)

# Turn 1
r1 = await model_multi.ainvoke([HumanMessage(content="My name is Alice and my email is alice@example.com")])
print("Turn 1:", r1.content)

# Turn 2 — same session, tokens are reused
r2 = await model_multi.ainvoke([HumanMessage(content="What was my email again?")])
print("Turn 2:", r2.content)